In [1]:
print("Ok")

Ok


In [2]:
import os
from git import Repo
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_cohere import CohereEmbeddings, ChatCohere
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from dotenv import load_dotenv

In [3]:
import sys
print(sys.executable)

c:\Users\HP\.conda\envs\llmapp\python.exe


In [4]:
%pwd

'c:\\Auto-code-analyzer\\research'

In [5]:
!mkdir test_repo

In [6]:
try:
    repo.close()
except NameError:
    pass

In [7]:
import shutil, os, stat

def remove_readonly(func, path, exc_info):
    os.chmod(path, stat.S_IWRITE)
    func(path)

if os.path.exists("test_repo"):
    shutil.rmtree("test_repo", onerror=remove_readonly)

if os.path.exists("db"):
    shutil.rmtree("db", onerror=remove_readonly)

print("Cleanup done.")

Cleanup done.


In [8]:
repo_path = "test_repo/"
repo = Repo.clone_from("https://github.com/MalempatiGnapika/Auto-code-analyzer.git", to_path=repo_path)

In [9]:
loader = GenericLoader.from_filesystem(
    repo_path,
    glob="**/*",
    suffixes=[".py"],
    parser=LanguageParser(language=Language.PYTHON, parser_threshold=500),
)
documents = loader.load()
documents

[Document(metadata={'source': 'test_repo\\app.py', 'language': <Language.PYTHON: 'python'>}, page_content='from langchain.vectorstores import Chroma\nfrom src.helper import load_embedding\nfrom dotenv import load_dotenv\nimport os\nfrom src.helper import repo_ingestion\nfrom flask import Flask, render_template, jsonify, request\nfrom langchain.llms import Cohere\nfrom langchain.memory import ConversationSummaryMemory\nfrom langchain.chains import ConversationalRetrievalChain\n\napp = Flask(__name__)\n\nload_dotenv()\n\nCOHERE_API_KEY = os.environ.get(\'COHERE_API_KEY\')\nos.environ["COHERE_API_KEY"] = COHERE_API_KEY\n\nembeddings = load_embedding()\npersist_directory ="db" \n# Now we can load the persisted database from disk, and use it as normal.\nvectordb = Chroma(persist_directory=persist_directory,\n                  embedding_function=embeddings)\n\nllm = Cohere(model="command", temperature=0.4)\nmemory = ConversationSummaryMemory(llm=llm, memory_key = "chat_history", return_messa

In [10]:
print(len(documents))
for doc in documents:
    print(doc.metadata["source"])

6
test_repo\app.py
test_repo\setup.py
test_repo\store_index.py
test_repo\template.py
test_repo\src\helper.py
test_repo\src\__init__.py


In [11]:
for doc in documents:
    print(doc.metadata["source"], "-", len(doc.page_content), "characters")

test_repo\app.py - 1725 characters
test_repo\setup.py - 238 characters
test_repo\store_index.py - 464 characters
test_repo\template.py - 807 characters
test_repo\src\helper.py - 1103 characters
test_repo\src\__init__.py - 0 characters


In [12]:
documents_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=500, chunk_overlap=20
)
texts = documents_splitter.split_documents(documents)
len(texts)

12

In [13]:
print("Documents loaded:", len(documents))
print("Chunks created:", len(texts))
for t in texts:
    print(t.metadata["source"])

Documents loaded: 6
Chunks created: 12
test_repo\app.py
test_repo\app.py
test_repo\app.py
test_repo\app.py
test_repo\app.py
test_repo\setup.py
test_repo\store_index.py
test_repo\template.py
test_repo\template.py
test_repo\src\helper.py
test_repo\src\helper.py
test_repo\src\helper.py


In [14]:
load_dotenv()
COHERE_API_KEY = os.environ.get("COHERE_API_KEY")
os.environ["COHERE_API_KEY"] = COHERE_API_KEY

In [15]:
embeddings = CohereEmbeddings(model="embed-english-v3.0")

In [16]:
vectordb = Chroma.from_documents(texts, embedding=embeddings, persist_directory="./db")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [17]:
print(vectordb._collection.count())


12


In [18]:
llm = ChatCohere(model="command-a-03-2025", temperature=0.4)

In [19]:
contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Given a chat history and the latest user question, formulate a standalone question. Do NOT answer it, just reformulate or return as-is."),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)
retriever = vectordb.as_retriever(search_type="mmr", search_kwargs={"k": 3})
history_aware_retriever = create_history_aware_retriever(llm, retriever, contextualize_q_prompt)

In [20]:
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an assistant for question-answering tasks. Use the retrieved context to answer concisely.\n\n{context}"),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)
question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

In [21]:
chat_history = []

In [22]:
question = "what is the repo_ingestion function doing?"
result = rag_chain.invoke({"input": question, "chat_history": chat_history})
chat_history.extend([HumanMessage(content=question), AIMessage(content=result["answer"])])
print(result["answer"])

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Number of requested results 20 is greater than number of elements in index 12, updating n_results = 12


The `repo_ingestion` function is responsible for cloning a Git repository from a given URL and saving it to a local directory named "repo". Here’s a breakdown of what it does:

1. **Create Directory**: It ensures that a directory named "repo" exists by using `os.makedirs("repo", exist_ok=True)`. If the directory already exists, it won't raise an error.
2. **Clone Repository**: It uses the `Repo.clone_from` method from the `git` library to clone the repository specified by `repo_url` into the "repo" directory.

In summary, the function fetches and stores a Git repository locally for further processing or analysis.


In [23]:
import cohere

co = cohere.ClientV2(api_key=os.environ["COHERE_API_KEY"])
models = co.models.list()
for m in models.models:
    print(m.name, "-", m.endpoints)

c4ai-aya-expanse-32b - ['generate', 'chat']
c4ai-aya-vision-32b - ['chat']
cohere-transcribe-03-2026 - ['transcriptions']
command-a-03-2025 - ['chat']
command-a-plus-05-2026 - ['generate', 'chat']
command-a-reasoning-08-2025 - ['chat']
command-a-translate-08-2025 - ['chat']
command-a-vision-07-2025 - ['chat']
command-r-08-2024 - ['generate', 'chat', 'summarize']
command-r-plus-08-2024 - ['generate', 'chat', 'summarize']
command-r7b-12-2024 - ['generate', 'chat']
command-r7b-arabic-02-2025 - ['generate', 'chat']
embed-english-light-v3.0 - ['embed']
embed-english-light-v3.0-image - ['embed_image']
embed-english-v3.0 - ['embed']
embed-english-v3.0-image - ['embed_image']
embed-multilingual-light-v3.0 - ['embed']
embed-multilingual-light-v3.0-image - ['embed_image']
embed-multilingual-v3.0 - ['embed']
embed-multilingual-v3.0-image - ['embed_image']


In [24]:
question = "what is the text_splitter function doing?"
result = rag_chain.invoke({"input": question, "chat_history": chat_history})
chat_history.extend([HumanMessage(content=question), AIMessage(content=result["answer"])])
print(result["answer"])

Number of requested results 20 is greater than number of elements in index 12, updating n_results = 12


The `text_splitter` function is designed to split a list of documents into smaller chunks of text. Here’s what it does in detail:

1. **Initialize Text Splitter**: It uses `RecursiveCharacterTextSplitter.from_language` to create a text splitter specifically configured for the Python language. This splitter is set up with:
   - `chunk_size=500`: Each chunk will be approximately 500 characters long.
   - `chunk_overlap=20`: There will be an overlap of 20 characters between consecutive chunks to ensure context continuity.

2. **Split Documents**: The `split_documents` method is called on the text splitter, passing the `documents` list as input. This method processes the documents and splits them into smaller chunks based on the configured parameters.

3. **Return Text Chunks**: The function returns the list of text chunks, which are smaller segments of the original documents.

In summary, the `text_splitter` function breaks down large documents into manageable chunks while maintaining som

In [25]:
question = "what does the load_embedding function return?"
result = rag_chain.invoke({"input": question, "chat_history": chat_history})
chat_history.extend([HumanMessage(content=question), AIMessage(content=result["answer"])])
print(result["answer"])

Number of requested results 20 is greater than number of elements in index 12, updating n_results = 12


The `load_embedding` function returns an instance of `CohereEmbeddings` configured with the model `"embed-english-v3.0"`. This instance is used to generate embeddings for text data, which are numerical representations of text that capture semantic meaning. These embeddings can be used for tasks like similarity search, clustering, or as input to machine learning models.

In summary, the function returns an embedding model ready to convert text into vector representations.


In [26]:
question = "what does the /chatbot route in app.py do?"
result = rag_chain.invoke({"input": question, "chat_history": chat_history})
chat_history.extend([HumanMessage(content=question), AIMessage(content=result["answer"])])
print(result["answer"])

Number of requested results 20 is greater than number of elements in index 12, updating n_results = 12


The `/chatbot` route in `app.py` is not explicitly defined in the provided code snippet. However, based on the context and the `chat` function, it appears that the `/chatbot` route is likely handled by the `chat` function, which processes user input and generates a response.

Here’s what the `chat` function does:

1. **Retrieve User Input**: It retrieves the user's message (`msg`) from the request form.
2. **Process Input**: If the input is `"clear"`, it deletes the "repo" directory using `os.system("rm -rf repo")`.
3. **Generate Response**: It calls the `qa` function (not shown in the snippet) with the user's input to generate a response.
4. **Print and Return Response**: It prints the generated response and returns it as a string.

If the `/chatbot` route is indeed mapped to this `chat` function (e.g., via `@app.route('/chatbot', methods=['POST'])`), it would handle POST requests, process the user's message, and return the chatbot's response.

**Example Route Definition (Not in Provi

In [27]:
question = "how does the app clear the cloned repository?"
result = rag_chain.invoke({"input": question, "chat_history": chat_history})
chat_history.extend([HumanMessage(content=question), AIMessage(content=result["answer"])])
print(result["answer"])

Number of requested results 20 is greater than number of elements in index 12, updating n_results = 12


The app clears the cloned repository by checking if the user input is the string `"clear"`. If it is, the app executes a shell command to delete the "repo" directory. Here’s how it works in detail:

1. **Check Input**: In the `chat` function, the user's input (`msg`) is checked against the string `"clear"`:
   ```python
   if input == "clear":
       os.system("rm -rf repo")
   ```

2. **Delete Directory**: If the input matches `"clear"`, the `os.system` function is used to execute the shell command `rm -rf repo`. This command recursively deletes the "repo" directory and all its contents.

**Explanation of the Command**:
- `rm`: Removes files or directories.
- `-r` (or `--recursive`): Removes directories and their contents recursively.
- `-f` (or `--force`): Ignores non-existent files and does not prompt for confirmation.

In summary, when the user sends `"clear"` as input, the app deletes the entire "repo" directory, effectively clearing the cloned repository.
